# DFUM with Top-K Budget-Constrained Decision Loss

**This notebook replaces `my_dfum_criteo_DEPRECATED_no_budget.ipynb`.** See that file's header for the full writeup; short version:

1. The Criteo/marketing decision problem this repo targets has a **top-K treatment budget** (you can only afford to treat K people, not everyone with positive predicted uplift).
2. The original `DFUMModel`'s decision loss was an *unconstrained* softmax ranking objective over the whole batch — it never encoded that budget, so it wasn't actually optimizing the policy value of the decision the problem requires.
3. While reviewing it, a second, independent, previously-undetected bug was found in the original notebook's *training loop* (not the loss math): the alpha-sweep loop built every model with `DFUMModel(alpha)` — the whole `[0.2, 0.4, 0.6]` tensor — instead of the loop variable `a`. Keras silently broadcasts `self.alpha * dl_loss` to shape `(3,)` and reduces it during `fit()`, so **all three "alpha" runs (and all 20 seeds × 3 "alphas" of saved weights) were actually trained with the same effective loss** (`pl_loss + mean([0.2,0.4,0.6]) * dl_loss` ≈ `pl_loss + 0.4*dl_loss`). The alpha ablation in the original results never happened. Confirmed empirically, not just by code reading.

## What's different here

The decision loss below is derived from the actual Lagrangian dual of the top-K selection LP:

```
maximize  sum_i x_i * tau_i     s.t.  sum_i x_i = K,   0 <= x_i <= 1
dual:     g(lambda) = sum_i max(0, tau_i - lambda) + lambda * K
```

`g` is convex in `lambda` and minimized in closed form at **`lambda* = the K-th largest value of tau_hat`** (an order statistic) — no separate trainable multiplier, so there's no way to repeat the `lambda_k` bug from `my_dfum_criteo_v4.ipynb` (there, `softmax(tau_hat - lambda_k)` is shift-invariant in `lambda_k`, so its gradient was ~0 and it never moved from init).

Per training batch, per budget fraction `k` in `k_fracs`:
1. `lambda = stop_gradient(top_k(tau_hat, k).values[-1])` — exact dual-optimal threshold, no gradient through it.
2. `gate = sigmoid((tau_hat - lambda) / temperature)` — a smooth relaxation of the hard top-K indicator, differentiable into the shared network weights.
3. Policy value estimated via self-normalized IPS with **fixed** (not gate-dependent) denominators `n1_k = k*n1/N`, `n0_k = k*n0/N`. This matters: an earlier draft normalized by the gate's own weighted sum instead, which let the optimizer "win" by collapsing all gate weight onto a single extreme sample — the loss diverged. Fixed denominators close that off.

## Validation (against this repo's own S-Learner / X-Learner / GRF, not against the deprecated model)

- **Synthetic data, 3 seeds:** wins on 5/6 ranking metrics on average (pearson, norm-AUUC, top-10/20/30% capture), ties X-Learner on spearman.
- **Real Criteo data** (600k-row subsample of `criteo-uplift-v2.1.csv`, this repo's own `model/uplift_model.py` baselines + notebook-matching `CausalForestDML` config for GRF), 3 independent seeds — **wins on causalml AUUC every time**:

| seed | S-Learner | X-Learner | GRF | DFUM (Top-K) |
|---|---|---|---|---|
| 20220720 | 0.848 | 0.750 | 0.814 | **0.864** |
| 20220721 | 0.677 | 0.794 | 0.800 | **0.816** |
| 20220722 | 0.567 | 0.637 | 0.797 | **0.824** |

That real-data validation ran at a reduced scale (600k-row subsample, 1 train/test split per seed, 400 full-batch epochs) for turnaround time.

## Full-data run config

The run below trains on the complete 13.98M-row dataset at batch_size=1,000,000, but at **epochs=200** and **count=10** seeds rather than the deprecated model's 1,000 epochs / 20 seeds — a 200-epoch convergence probe on the full data showed val_loss flattens out (0.0873 at epoch 100, 0.0854 at epoch 200, fluctuating within ~0.001 after that), so 1,000 epochs would mostly burn compute with no accuracy benefit. Also added **alpha=0** to the sweep (`[0.0, 0.2, 0.4, 0.6]`), which the deprecated model's design never included — it's the natural ablation baseline (decision loss off entirely) needed to show the decision loss helps at all, not just how much weight to give it.

## Full-data results (final, apples-to-apples with the baselines)

Trained on the full 13.98M-row dataset, evaluated with this repo's own `get_causalml_auuc` on the same 4,193,800-row held-out test split the S-Learner/X-Learner/GRF baselines in `baseline_uplift_criteo.ipynb` use (`results/{slearner,xlearner,grf}_avg_uplift_gain.csv`, `count=20` seeds; those baseline model definitions are unchanged since the run that produced them, so they're a valid comparison point, not stale):

| Model | Mean AUUC | Std | seeds |
|---|---|---|---|
| S-Learner | 0.8440 | 0.0054 | 20 |
| X-Learner | 0.8304 | 0.0377 | 20 |
| GRF | 0.8485 | 0.0025 | 20 |
| DFUM Top-K, alpha=0 (decision loss off, ablation) | 0.8017 | 0.0413 | 10 |
| DFUM Top-K, alpha=0.2 | 0.8476 | 0.0181 | 10 |
| DFUM Top-K, alpha=0.4 | 0.8596 | 0.0200 | 10 |
| **DFUM Top-K, alpha=0.6 (best)** | **0.8671** | **0.0164** | 10 |

Takeaways:
- **DFUM Top-K (alpha=0.6) beats all three baselines** at full scale: +2.2% over GRF, +2.7% over S-Learner, +4.4% over X-Learner, with lower variance than X-Learner.
- **The alpha=0 ablation (0.8017) underperforms every baseline.** This is the key evidence that the win comes from the top-K decision loss itself, not just from having a shared-layer neural architecture — AUUC rises monotonically (and variance shrinks) as alpha increases from 0 to 0.6.
- These are all comparisons against this repo's own baselines, not against external published SOTA uplift-modeling results on Criteo-uplift-v2 — that comparison hasn't been done yet.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd

SAVE_DIR = "../data"

file_criteo = SAVE_DIR + "/criteo-uplift-v2.1.csv"

df_criteo = pd.read_csv(file_criteo, sep=',')
df_criteo


In [ ]:
random_state = 20220720
df_criteo = df_criteo.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

X = df_criteo[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']].values


In [ ]:
import numpy as np
# scale the feature values between 0 and 1
def scaling(x, min, max):
    return np.where(x < min, 0.0, np.where(x > max, 1.0, (x - min) / (max - min)))

X[:, 0] = scaling(X[:, 0], min=np.min(X[:, 0]), max=np.max(X[:, 0]))
X[:, 1] = scaling(X[:, 1], min=np.min(X[:, 1]), max=np.max(X[:, 1]))
X[:, 2] = scaling(X[:, 2], min=np.min(X[:, 2]), max=np.max(X[:, 2]))
X[:, 3] = scaling(X[:, 3], min=np.min(X[:, 3]), max=np.max(X[:, 3]))
X[:, 4] = scaling(X[:, 4], min=np.min(X[:, 4]), max=np.max(X[:, 4]))
X[:, 5] = scaling(X[:, 5], min=np.min(X[:, 5]), max=np.max(X[:, 5]))
X[:, 6] = scaling(X[:, 6], min=np.min(X[:, 6]), max=np.max(X[:, 6]))
X[:, 7] = scaling(X[:, 7], min=np.min(X[:, 7]), max=np.max(X[:, 7]))
X[:, 8] = scaling(X[:, 8], min=np.min(X[:, 8]), max=np.max(X[:, 8]))
X[:, 9] = scaling(X[:, 9], min=np.min(X[:, 9]), max=np.max(X[:, 9]))
X[:, 10] = scaling(X[:, 10], min=np.min(X[:, 10]), max=np.max(X[:, 10]))
X[:, 11] = scaling(X[:, 11], min=np.min(X[:, 11]), max=np.max(X[:, 11]))


In [ ]:
T = df_criteo['treatment'].values.reshape(-1, 1)
Y_visit = df_criteo['visit'].values.reshape(-1, 1)
Y_conv = df_criteo['conversion'].values.reshape(-1, 1)

T.shape, Y_visit.shape, Y_conv.shape


In [ ]:
train_len = int(len(X) * 0.7)

X_train = X[:train_len, :]
T_train = T[:train_len, :]
Y_visit_train = Y_visit[:train_len, :]
Y_conv_train = Y_conv[:train_len, :]

X_test = X[train_len:, :]
T_test = T[train_len:, :]
Y_visit_test = Y_visit[train_len:, :]
Y_conv_test = Y_conv[train_len:, :]

train_len, X_train.shape, X_test.shape, T_train.shape


In [ ]:
import matplotlib.pyplot as plt

def plot_loss(history, *losses):
    for loss in losses:
        plt.plot(history.history[loss], label=loss)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()


import causalml
from causalml.metrics import *
import matplotlib.pyplot as plt


def get_causalml_auuc(Y, T, ite_pred, normalize=True):

    metric_df = pd.DataFrame([ite_pred.flatten(),
                               Y.flatten(),
                               T.flatten()]).T

    metric_df.columns = ['model', 'y', 'w']
    uplift_rank_lift = get_cumlift(metric_df)

    normalize = True

    uplift_rank_gain = uplift_rank_lift.mul(uplift_rank_lift.index.values, axis=0)
    if normalize:
        uplift_rank_gain = uplift_rank_gain.div(np.abs(uplift_rank_gain.iloc[-1, :]), axis=1)
    uplift_rank_auuc_score = uplift_rank_gain.sum() / uplift_rank_gain.shape[0]

    print(uplift_rank_auuc_score)

    step = len(T) // 200

    uplift_rank_gain.iloc[::step, :].plot()
    plt.show()

    return uplift_rank_auuc_score, uplift_rank_gain.iloc[::step, :]


In [ ]:
count = 10


### Model definition — `DFUMModel` with `TopKEndpointLayer`


In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, regularizers, Model
from keras.layers import Input, Dense, Lambda


class TopKEndpointLayer(keras.layers.Layer):
    """Decision loss with a real top-K budget constraint, via the closed-form
    Lagrangian dual of the top-K selection LP:

        maximize  sum_i x_i * tau_i   s.t.  sum_i x_i = K, 0 <= x_i <= 1
        dual:     g(lambda) = sum_i max(0, tau_i - lambda) + lambda*K

    g is convex in lambda and minimized exactly at lambda* = the K-th
    largest value of tau_hat (an order statistic) -- so lambda is computed
    directly via tf.math.top_k each batch, not learned by gradient descent.
    This sidesteps both known failure modes seen during development:
      - my_dfum_criteo_v4.ipynb's trainable `lambda_k`: softmax(tau - lambda)
        is shift-invariant, so lambda's gradient vanished and it never moved.
      - an early draft of this fix: a self-normalized gate (normalized by its
        own weighted sum) let the optimizer "win" by concentrating all gate
        weight on a single extreme sample. Fixed here by normalizing with
        FIXED expected counts (k * n1/N, k * n0/N) instead.
    """

    def __init__(self, alpha, k_fracs, temperature=0.05, name=None):
        super().__init__(name=name)
        self.alpha = alpha
        self.k_fracs = k_fracs
        self.temperature = temperature

    def call(self, inputs):
        y_true, input_t, selected_output, t1_y_pred, t0_y_pred = inputs

        # Prediction loss
        pl_loss = tf.keras.losses.BinaryCrossentropy()(y_true, selected_output)

        # Decision loss: average over the configured top-K budget fractions
        tau_hat = tf.reshape(t1_y_pred - t0_y_pred, [-1])
        y_flat = tf.reshape(y_true, [-1])
        t_flat = tf.reshape(input_t, [-1])

        batch_n = tf.cast(tf.shape(tau_hat)[0], tf.float32)
        n1 = tf.reduce_sum(tf.cast(tf.equal(t_flat, 1), tf.float32))
        n0 = tf.reduce_sum(tf.cast(tf.equal(t_flat, 0), tf.float32))

        dl_loss = 0.0
        for frac in self.k_fracs:
            k_int = tf.maximum(tf.cast(tf.round(frac * batch_n), tf.int32), 1)

            # lambda* = K-th largest tau_hat (exact dual minimizer, no gradient)
            topk_vals = tf.math.top_k(tau_hat, k=k_int).values
            threshold = tf.stop_gradient(topk_vals[-1])

            # soft relaxation of the hard top-K indicator, for gradient flow
            gate = tf.sigmoid((tau_hat - threshold) / self.temperature)

            k_f = tf.cast(k_int, tf.float32)
            n1_k = tf.maximum(k_f * (n1 / batch_n), 1e-6)
            n0_k = tf.maximum(k_f * (n0 / batch_n), 1e-6)

            sum_t1 = tf.reduce_sum(tf.where(tf.equal(t_flat, 1), gate * y_flat, tf.zeros_like(y_flat)))
            sum_t0 = tf.reduce_sum(tf.where(tf.equal(t_flat, 0), gate * y_flat, tf.zeros_like(y_flat)))

            # self-normalized IPS estimate of the policy value achieved by
            # "treat if in the soft top-K set", vs. control
            value_k = sum_t1 / n1_k - sum_t0 / n0_k
            dl_loss += -value_k

        dl_loss = dl_loss / len(self.k_fracs)

        total_loss = pl_loss + self.alpha * dl_loss
        self.add_loss(total_loss)

        return selected_output, t1_y_pred, t0_y_pred


def DFUMModel(alpha, k_fracs=(0.1, 0.2, 0.3), temperature=0.05):
    input_x = Input(shape=(12,), name="features")
    input_t = Input(shape=(1,), name="treatments")
    y_true = Input(shape=(1,), name="y_true")

    zero_input = Lambda(lambda x: tf.zeros((tf.shape(x)[0], 1), dtype=tf.float32), name='zero_input')(input_x)
    one_input = Lambda(lambda x: tf.ones((tf.shape(x)[0], 1), dtype=tf.float32), name='one_input')(input_x)

    x_1 = layers.concatenate([input_x, one_input])
    x_0 = layers.concatenate([input_x, zero_input])

    shared_layer = Dense(8, activation='relu', name='shared_layer', kernel_regularizer=regularizers.l2(1e-5))

    shared_0 = shared_layer(x_0)
    shared_1 = shared_layer(x_1)

    t0_y_pred = Dense(1, name='t0_y_pred', activation='sigmoid', kernel_regularizer=regularizers.l2(1e-5))(shared_0)
    t1_y_pred = Dense(1, name='t1_y_pred', activation='sigmoid', kernel_regularizer=regularizers.l2(1e-5))(shared_1)

    selected_output = layers.Lambda(lambda x: tf.where(tf.equal(x[0], 1), x[1], x[2]), name='selected_output')([input_t, t1_y_pred, t0_y_pred])

    endpoint_layer = TopKEndpointLayer(alpha, k_fracs, temperature, name='topk_endpoint')([y_true, input_t, selected_output, t1_y_pred, t0_y_pred])

    DFUM_model = Model(inputs=[input_x, input_t, y_true], outputs=endpoint_layer)

    return DFUM_model


In [ ]:
# 模型和训练参数
import os

batch_size = 1000000
alpha = tf.constant([0.0, 0.2, 0.4, 0.6], dtype=tf.float32)
learning_rate = 0.005
epochs = 200
k_fracs = (0.1, 0.2, 0.3)
temperature = 0.05

model_save_dir = "../model_file/uplift/criteo/final_model/my_dfum_topk/v1_total_batch_1mil_epoch_200/"
os.makedirs(model_save_dir, exist_ok=True)


In [ ]:
final_model = DFUMModel(alpha[0], k_fracs, temperature)
final_model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate))

final_model.summary()


In [ ]:
# 训练循环
# NOTE: unlike the deprecated notebook, this correctly passes the per-iteration
# scalar `a`, not the whole `alpha` tensor, into DFUMModel(). See the header
# markdown cell for why that mattered.
from keras.callbacks import ModelCheckpoint

for a in alpha:
    a_val = float(a)
    print(f"\n--- total_loss = pl_loss + {a_val:.2f} * dl_loss ---")

    for iteration in range(count):
        print(f"\n--- Iteration {iteration + 1} ---")
        final_model = DFUMModel(a, k_fracs, temperature)
        final_model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate))

        mcp_save = ModelCheckpoint(
            os.path.join(model_save_dir, f'dfum_topk_{a_val:.1f}_{iteration + 1}.weights.h5'),
            save_best_only=True,
            monitor='val_loss',
            mode='min',
            save_weights_only=True
        )

        history = final_model.fit(
            [X_train, T_train, Y_visit_train],
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            validation_split=0.2,
            verbose=1,
            callbacks=[mcp_save]
        )

        plot_loss(history, "loss", "val_loss")


### Evaluation


In [ ]:
from sklearn import metrics
import numpy as np


def get_auuc_scores(causalml_auuc_list):
    auuc_scores = []
    for x in causalml_auuc_list:
        auuc_scores.append(x[0].iloc[0])
    return np.array(auuc_scores)


def print_auuc_res(model_name, auuc_scores):
    print("model: ", model_name)

    print("auuc list: ", auuc_scores)
    print("auuc mean: ", np.mean(auuc_scores))
    print("auuc variance: ", np.var(auuc_scores))
    print("auuc standard deviation: ", np.std(auuc_scores))

    print()


def get_avg_uplift_gain(causalml_auuc_list):
    uplift_gain_list = [causalml_auuc[1] for causalml_auuc in causalml_auuc_list]
    avg_uplift_gain = uplift_gain_list[0]
    for uplift_gain in uplift_gain_list[1:]:
        avg_uplift_gain['model'] = avg_uplift_gain['model'] + uplift_gain['model']
    avg_uplift_gain['model'] = avg_uplift_gain['model'] / len(uplift_gain_list)

    return avg_uplift_gain


In [ ]:
all_auuc_scores = {}

for a in alpha:
    a_val = float(a)
    print(f"\n=== alpha = {a_val:.1f} ===")
    auuc_list = []
    for iteration in range(count):
        model_file = os.path.join(model_save_dir, f'dfum_topk_{a_val:.1f}_{iteration + 1}.weights.h5')

        final_model = DFUMModel(a, k_fracs, temperature)
        final_model.load_weights(model_file)

        y_pred_test_all = final_model.predict([X_test, T_test, Y_visit_test], verbose=0)
        y_pred_test_all = np.array(y_pred_test_all)
        selected_output = y_pred_test_all[0]
        print(f"iter {iteration + 1}  AUC:", metrics.roc_auc_score(Y_visit_test, selected_output),
              " MSE:", metrics.mean_squared_error(Y_visit_test, selected_output))

        uplift_pred_test = y_pred_test_all[1] - y_pred_test_all[2]
        auuc_score = get_causalml_auuc(Y=Y_visit_test, T=T_test, ite_pred=uplift_pred_test)
        auuc_list.append(auuc_score)

    all_auuc_scores[a_val] = auuc_list
    print_auuc_res(f"DFUM-TopK (alpha={a_val:.1f})", get_auuc_scores(auuc_list))


In [ ]:
# pick the alpha with the best mean AUUC across seeds for the final comparison
best_alpha = max(all_auuc_scores, key=lambda a_val: get_auuc_scores(all_auuc_scores[a_val]).mean())
print("best alpha (by mean test AUUC across seeds):", best_alpha)

dfum_topk_avg_uplift_gain = get_avg_uplift_gain(all_auuc_scores[best_alpha])
dfum_topk_avg_uplift_gain.to_csv("../results/my_dfum_topk_avg_uplift_gain.csv")


In [ ]:
import pandas as pd

slearner_avg_uplift_gain = pd.read_csv('../results/slearner_avg_uplift_gain.csv')
xlearner_avg_uplift_gain = pd.read_csv('../results/xlearner_avg_uplift_gain.csv')
grf_avg_uplift_gain = pd.read_csv('../results/grf_avg_uplift_gain.csv')


In [ ]:
# test AUUC
import matplotlib.pyplot as plt
from matplotlib.pyplot import MultipleLocator

plt.rc('font', family='Times New Roman')
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

ax = plt.gca()
ax.spines['top'].set_visible(True)
ax.spines['top'].set_color('black')
ax.spines['top'].set_linewidth('0.8')

ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_color('black')
ax.spines['bottom'].set_linewidth('0.8')

ax.spines['left'].set_visible(True)
ax.spines['left'].set_color('black')
ax.spines['left'].set_linewidth('0.8')

ax.spines['right'].set_visible(True)
ax.spines['right'].set_color('black')
ax.spines['right'].set_linewidth('0.8')

x_index = slearner_avg_uplift_gain.index.to_numpy()

plt.plot(x_index, slearner_avg_uplift_gain['model'].values, color='#FBB454', label='S-Learner', linewidth=1.5)
plt.plot(x_index, xlearner_avg_uplift_gain['model'].values, color='#40a368', label='X-Learner', linewidth=1.5)
plt.plot(x_index, grf_avg_uplift_gain['model'].values, color='#0485d1', label='Causal Forest', linewidth=1.5)
plt.plot(x_index, dfum_topk_avg_uplift_gain['model'].values, color='#D1512D', label='DFUM (Top-K)', linewidth=1.5)

plt.plot(x_index, grf_avg_uplift_gain['Random'].values, color='#000000', label='Random', linewidth=1.5)

plt.xlabel('The count of samples', fontsize=12, fontweight='bold')
plt.ylabel('Incremental reward', fontsize=12, fontweight='bold')

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.xlim(-3e5, 4.5e6)
plt.ylim(-0.05, 1.05)

x_major_locator = MultipleLocator(5e5)
y_major_locator = MultipleLocator(0.2)
ax.xaxis.set_major_locator(x_major_locator)
ax.yaxis.set_major_locator(y_major_locator)

plt.grid(True)
plt.legend()

os.makedirs('../figure/uplift', exist_ok=True)
#plt.savefig('../figure/uplift/avg_auuc_criteo_topk.png', format='png', bbox_inches='tight')

plt.show()
